# Phase 2 Pipeline: Minimal Instruction Set (MIS) Training

In Phase 2, we simulate execution traces of a formal MIS. 
We train Qwen2.5 on fixed-width state logs designed to be tokenization-invariant, effectively creating a granular, state-faithful world model.

In [ ]:
# Clone the GitHub repository and navigate into it so relative paths work perfectly in Colab
!git clone https://github.com/TinevimboMusingadi/lm-world-model.git
%cd lm-world-model

## 1. Assembly Trace Generation
The generator includes rigorous controls to prevent the **Constant PC Trap** where the LM simply counts upwards to fake execution. We introduce jumps and explicitly hold out certain Out-Of-Distribution (OOD) combinations.

In [ ]:
!python -m phase2_mis.generate_dataset --seed 42 --n 82000 --output data/splits/phase2_raw.jsonl
!python -m data.inspector --file data/splits/phase2_raw.jsonl

### Instruction Set Distributions
We can chart what instructions are primarily making up the generated dataset, to ensure balancing and that OOD holdouts were truly respected.

In [ ]:
import pandas as pd
import jsonlines
import plotly.express as px
import re

op_counts = []
splits = []
with jsonlines.open('data/splits/phase2_raw.jsonl') as f:
    for r in f:
        ops = re.findall(r'^\s*([A-Z]+)', r['code'], re.MULTILINE)
        op_counts.extend(ops)
        splits.append(r['split'])

df_ops = pd.DataFrame(op_counts, columns=['Instruction'])
fig = px.histogram(df_ops, x='Instruction', title='MIS Operator Frequency in Dataset',
                   color_discrete_sequence=['#636EFA'])
fig.show()

df_splits = pd.DataFrame(splits, columns=['Split'])
fig2 = px.histogram(df_splits, x='Split', title='Distribution of Programs by Split (Train, Val, Test In-Dist, Test OOD)')
fig2.show()

## 2. PEFT LoRA Training
We use the pre-configured training file to execute SFT on the MIS structured data.

In [ ]:
import yaml
from pathlib import Path

config = {
    "run_name": "phase2-qwen0.5b-condB-seed42",
    "phase": "phase2",
    "condition": "B",
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "data_file": "data/splits/phase2_raw.jsonl",
    "output_dir": "checkpoints/phase2_qwen0.5b_condB",
    "lora_r": 16,
    "lora_alpha": 32,
    "epochs": 3,
    "batch_size": 4,
    "grad_accum": 4,
    "lr": 2.0e-4,
    "max_seq_len": 1024
}
Path("training/configs").mkdir(parents=True, exist_ok=True)
with open("training/configs/phase2_qwen_0.5b.yaml", "w") as f:
    yaml.dump(config, f)

!python -m training.train_sft --config training/configs/phase2_qwen_0.5b.yaml

## 3. OOD Generalization Evaluation
Tests state trace fidelity (Levenshtein Distance) explicitly against OOD combinations.

In [ ]:
!python -m eval.evaluate --checkpoint checkpoints/phase2_qwen0.5b_condB \
                         --data data/splits/phase2_raw.jsonl \
                         --split test_indist \
                         --condition B \
                         --temperature 0.0

!python -m eval.evaluate --checkpoint checkpoints/phase2_qwen0.5b_condB \
                         --data data/splits/phase2_raw.jsonl \
                         --split test_ood \
                         --condition B \
                         --temperature 0.0

### Visualising Formal OOD Capabilities
Graphing the Levenshtein Distances and Error Step divergences.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    df_res = pd.read_csv("results/eval_results.csv")
    
    fig = make_subplots(rows=1, cols=2, subplot_titles=("In-Dist vs OOD Trace Accuracy", "Levenshtein Distance Penalty Gap"))
    
    fig.add_trace(
        go.Bar(x=df_res['split'], y=df_res['trace_accuracy'], marker_color="#1f77b4"),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(x=df_res['split'], y=df_res['levenshtein_distance'], marker_color="#d62728"),
        row=1, col=2
    )
    fig.update_layout(showlegend=False, title_text="Phase 2 OOD Testing Fidelity")
    fig.show()
except FileNotFoundError:
    print("Run evaluate scripts first to generate stats.")